# Create Subsets for Evaluation (Plain Python)

## Import packages

In [1]:
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_colwidth", 200)

## Helper functions

In [2]:
def load_dataframe(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)

    raise ValueError(f"Unsupported input file type: {suffix}")


def save_dataframe(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        df.to_csv(path, index=False)
        return
    if suffix in {".parquet", ".pq"}:
        df.to_parquet(path, index=False)
        return
    if suffix == ".jsonl":
        df.to_json(path, orient="records", lines=True)
        return
    if suffix == ".json":
        df.to_json(path, orient="records")
        return
    if suffix in {".pkl", ".pickle"}:
        df.to_pickle(path)
        return

    raise ValueError(f"Unsupported output file type: {suffix}")


def as_text_series(series):
    return series.fillna("").astype(str)


def select_longest_unique_start_rows(df, column, n, start_word_count):
    sorted_df = (
        df.assign(
            _target_text=as_text_series(df[column]).str.strip(),
        )
        .assign(_target_len=lambda frame: frame["_target_text"].str.len())
        .sort_values("_target_len", ascending=False, kind="mergesort")
        .drop(columns=["_target_text", "_target_len"])
    )

    selected_indices = []
    existing_start_words = set()

    for idx, value in as_text_series(sorted_df[column]).items():
        start_words = value.split()[:start_word_count]
        start_words = " ".join(start_words)

        if start_words in existing_start_words:
            continue

        existing_start_words.add(start_words)
        selected_indices.append(idx)

        if len(selected_indices) >= n:
            break

    return sorted_df.loc[selected_indices].copy()


def sample_records_by_group(
    df,
    group_col,
    sample_size,
    random_state=123,
    allow_smaller_groups=False,
):
    group_sizes = df[group_col].value_counts(dropna=False)
    too_small_groups = group_sizes[group_sizes < sample_size]

    if not allow_smaller_groups and not too_small_groups.empty:
        raise ValueError(
            "At least one group has fewer than "
            f"{sample_size} rows. Set allow_smaller_groups=True "
            "to keep all rows from smaller groups."
        )

    def sample_group(group):
        n = (
            min(sample_size, len(group))
            if allow_smaller_groups
            else sample_size
        )
        return group.sample(n=n, random_state=random_state)

    return (
        df.groupby(group_col, group_keys=False, dropna=False)
        .apply(sample_group)
        .reset_index(drop=True)
    )

## Set parameters

In [ ]:
# Replace these paths and column names for your local data.
data_path = Path("data/filtered_output.csv")
target_col = "text"

data = load_dataframe(data_path)
display(data.head())
print(f"Loaded {len(data):,} rows")

In [4]:
# # Replace these paths and column names for your local data.
# data_path = Path("path/to/input.csv")
# target_col = "MY_TARGET"

# data = load_dataframe(data_path)
# display(data.head())
# print(f"Loaded {len(data):,} rows")


In [5]:
# N Longest Records with X Unique Start Words parameters.
N = 10
X = 3
output_path_longest = Path("data/longest_unique_start_records.csv")

In [6]:
# # N Longest Records with X Unique Start Words parameters.
# N = 1000
# X = 3
# output_path_longest = Path("outputs/longest_unique_start_records.csv")


In [7]:
# M Random Records from each Reporting Unit parameters.
M = 5
unit_col = "hc_id"
random_state = 123
allow_smaller_groups = False
output_path_random = Path("data/random_records_by_unit.csv")

In [8]:
# # M Random Records from each Reporting Unit parameters.
# M = 50
# unit_col = "MY_UNIT_COL"
# random_state = 123
# allow_smaller_groups = False
# output_path_random = Path("outputs/random_records_by_unit.csv")


## _N_ Longest Records with _X_ Unique Start Words

In [ ]:
longest_unique_start_df = select_longest_unique_start_rows(
    data,
    column=target_col,
    n=N,
    start_word_count=X,
)

display(longest_unique_start_df.head())
print(
    f"Selected {len(longest_unique_start_df):,} records "
    f"with unique first {X} words"
)

In [ ]:
save_dataframe(longest_unique_start_df, output_path_longest)
print(
    f"Saved {len(longest_unique_start_df):,} rows "
    f"to {output_path_longest.resolve()}"
)

## _M_ Random Records from each Reporting Unit

In [ ]:
# Set allow_smaller_groups=True if some groups have fewer than M records.
rand_strat_sample_df = sample_records_by_group(
    data,
    group_col=unit_col,
    sample_size=M,
    random_state=random_state,
    allow_smaller_groups=allow_smaller_groups,
)

display(rand_strat_sample_df.head())
print(f"Selected {len(rand_strat_sample_df):,} stratified random records")

In [ ]:
save_dataframe(rand_strat_sample_df, output_path_random)
print(
    f"Saved {len(rand_strat_sample_df):,} rows "
    f"to {output_path_random.resolve()}"
)

## Create Other Subsets as Suitable for Your Case